# TCO-Analyse: Elektro-LKW vs. Diesel
## Einsatzbereich Regional- und Verteilerverkehr (<500 km/Tag)

Dieses Notebook modelliert die Total Cost of Ownership (TCO) für drei Antriebsarten
über eine typische Nutzungsdauer von 8 Jahren.

---

## 1. Konzeptioneller Rahmen

Die TCO eines LKW setzt sich zusammen aus:

$$
\text{TCO} = K_{Anschaffung} + \sum_{t=1}^{T} \frac{K_{Energie}(t) + K_{Wartung}(t) + K_{Maut}(t) + K_{Versicherung}}{(1+r)^t} - K_{Restwert}
$$

wobei $r$ der Diskontierungssatz und $T$ die Nutzungsdauer in Jahren ist.

### 1.1 Energiekosten

**Diesel:**
$$
K_{Energie,D}(t) = V_{D} \cdot P_{Diesel}(t) \cdot d_{Jahr}
$$

**Elektro (BEV):**
$$
K_{Energie,E}(t) = V_{E} \cdot P_{Strom}(t) \cdot d_{Jahr}
$$

wobei $V$ der Verbrauch pro 100 km, $P$ der Energiepreis und $d_{Jahr}$ die Jahreskilometerleistung.

### 1.2 Mautkosten

Für E-LKW gilt ab 2026 ein reduzierter Mautsatz von 75% Rabatt auf den Infrastrukturanteil.
Ab 2026 zahlt ein E-LKW ca. 25% der vollen Maut:

$$
K_{Maut,E}(t) = K_{Maut,D}(t) \cdot \alpha_{E}(t)
$$

### 1.3 Break-Even

Der Break-Even-Punkt ist das Jahr $t^*$, in dem die kumulierten TCO beider Varianten gleich sind:

$$
\sum_{t=0}^{t^*} K_{Gesamt,E}(t) = \sum_{t=0}^{t^*} K_{Gesamt,D}(t)
$$

---

## 2. Serienmodelle (Stand Mai 2026)

| Hersteller | Modell | Segment | Reichweite | Batterie | Ladezeit (MCS) | Status |
|---|---|---|---|---|---|---|
| **Mercedes-Benz** | eActros 600 | Fernverkehr/Regional | ~500 km | 621 kWh (LFP) | ~30 min | ✅ Serienproduktion |
| **MAN** | eTGX | Fernverkehr | ~570 km (Sattel) | bis 480 kWh (NMC) | ~27 min | ✅ Serienfertigung ab 06/2025 |
| **MAN** | eTGS | Regional/Bau | bis 820 km/Tag | bis 480 kWh | ~27 min | ✅ Serienfertigung ab 06/2025 |
| **MAN** | eTGL | Urban/Verteiler | ~235 km/Tag | bis 160 kWh | 1,5h (150kW DC) | ✅ Verfügbar |
| **Volvo** | FH Electric | Regional/Überlandverkehr | ~300 km | 540 kWh | – | ✅ Seit 2022 |
| **Volvo** | FH Aero Electric | Fernverkehr | ~600 km | 780 kWh | <40 min | 🔜 Bestellstart Q2/2026 |
| **Volvo** | FM Electric | Regional/schwer | ~300 km | 540 kWh | – | ✅ Verfügbar |
| **Volvo** | FL Electric | Urban | ~300 km | variabel | – | ✅ Verfügbar |
| **Scania** | 45 R/S BEV | Regional | bis 560 kWh | 240–560 kWh | MCS ab 2026 | ✅ Verfügbar |
| **Scania** | 25 P/L BEV | Urban/Verteiler | ~300 km | 320 kWh | – | ✅ Verfügbar |
| **DAF** | XF Electric | Fernverkehr | ~500 km | – | – | ✅ In Auslieferung |
| **DAF** | XD Electric | Regional | ~300 km | – | – | ✅ Verfügbar |
| **Renault** | E-Tech T | Regional | ~300–500 km | bis 575 kWh | – | ✅ Verfügbar |
| **Tesla** | Semi | Fernverkehr | ~800 km (US) | 621 kWh (LFP) | ~30 min | ⚠️ US-Markt, EU unklar |

**Für sächsische Regional-/Verteilerlogistiker am relevantesten:**
MAN eTGS (Regional), Volvo FM Electric, Scania 45 R – alle in der 300–500 km Klasse,
gut geeignet für Hub-and-Spoke-Verkehre mit Depot-Laden.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ABHÄNGIGKEITEN
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        11,
})
print('Bibliotheken geladen.')

## 3. Parameter

**Quellen:**
- Anschaffungspreise: MAN, Daimler, Volvo (Herstellerangaben 2025)
- Energiepreise: EU Oil Bulletin (Diesel), BDEW Strompreisanalyse 2026
- Wartungskosten: TNO-Studie 2022, ICCT 2024
- Mautsätze: BALM (DE), Bundesfernstraßenmautgesetz 2024
- TCO-Vergleiche: ICCT 2023/2024, P3 Group 2024, ICT Institute 2025

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PARAMETER – hier anpassen
# ─────────────────────────────────────────────────────────────────────────────

# --- Nutzungsprofil ---
KM_JAHR          = 100_000   # Jahreskilometerleistung (km)
NUTZUNGSJAHRE    = 8         # Haltedauer (Jahre)
DISKONTIERUNG    = 0.04      # Kapitalkostensatz (4% üblich für Fuhrpark)
STARTJAHR        = 2025

# --- Anschaffungskosten (€, netto) ---
# Quelle: Herstellerangaben 2025; E-LKW = ca. 2–3× Diesel-Listpreis
KAUFPREIS = {
    'Diesel':  150_000,   # EURO VI Sattelzug 40t (z.B. MAN TGX, Volvo FH)
    'BEV':     380_000,   # E-LKW Regional 300–500 km (z.B. MAN eTGS, Volvo FM Electric)
    'BEV_Foerderung': 0,  # Aktuell kein KsNI mehr; ggf. Landesförderung einsetzen
    'Depot_Infrastruktur': 60_000,  # Ladeinfrastruktur (Trafo-Upgrade + Wallbox für 1 LKW-Anteil)
}

# --- Restwert nach Nutzungsdauer (% vom Kaufpreis) ---
RESTWERT_RATE = {
    'Diesel': 0.15,   # Gut etablierter Markt
    'BEV':    0.18,   # Unsicherer, aber Batterie hat Restwert
}

# --- Energiepreise 2025 (Startpunkt) ---
# Diesel: EU Oil Bulletin Durchschnitt DE (inkl. MwSt.)
# Strom: BDEW Gewerbestrompreis 2026 (netto, inkl. Netzentgelt)
ENERGIEPREIS_START = {
    'Diesel_EUR_L':  1.50,    # €/Liter Diesel (netto Gewerbe)
    'Strom_Depot':   0.18,    # €/kWh Industrie-/Gewerbestrom Depot (günstig)
    'Strom_Oeffentlich': 0.45, # €/kWh öffentliche Schnellladung (HPC)
    'Strom_PV':      0.08,    # €/kWh Eigenverbrauch PV-Anlage (bei Investition)
}

# Anteil der Ladung über Depot vs. öffentlich (muss = 1)
ANTEIL_DEPOT      = 0.85   # 85% Depot-Laden (gut planbare Routen)
ANTEIL_OEFFENTLICH = 0.15  # 15% unterwegs laden

# Effektiver Strompreis (gewichtetes Mittel)
STROM_EFFEKTIV = (ANTEIL_DEPOT * ENERGIEPREIS_START['Strom_Depot'] +
                  ANTEIL_OEFFENTLICH * ENERGIEPREIS_START['Strom_Oeffentlich'])

# --- Jährliche Energiepreissteigerungen ---
PREISSTEIGERUNG = {
    'Diesel': 0.04,   # ~4% p.a. (inkl. ETS2-Effekt ab 2027)
    'Strom':  0.01,   # ~1% p.a. (Strom strukturell günstiger werdend)
}

# --- Verbrauch ---
VERBRAUCH = {
    'Diesel_L_100km': 28.0,   # Liter/100km (typisch 40t Sattelzug)
    'BEV_kWh_100km':  97.0,   # kWh/100km (MAN eTruck Praxiswert 2025)
}

# --- Wartungskosten (€/km) ---
# Quelle: TNO 2022, ICCT 2024
# Diesel: Ölwechsel, AdBlue, Verschleißteile, Inspektionen
# BEV: deutlich weniger bewegliche Teile, keine Verbrennungsrückstände
WARTUNG_EUR_KM = {
    'Diesel': 0.10,   # 10 ct/km
    'BEV':    0.05,   # 5 ct/km (~50% weniger)
}

# --- Versicherung (€/Jahr, pauschal) ---
VERSICHERUNG = {
    'Diesel': 8_000,
    'BEV':    9_500,   # etwas höher wegen Batterie-Vollkasko
}

# --- Mautsätze (ct/km) – DE-Netz, Anteil mautpflichtig ---
# Basis: BALM 2025; E-LKW ab 2026: ~25% der vollen Maut (75% Rabatt auf Infrastrukturteil)
MAUT_START_CT_KM = {
    'Diesel': 50.0,   # ct/km, EURO VI, 40t
    'BEV':    12.5,   # ct/km (25% von 50 ct)
}
ANTEIL_MAUTPFLICHTIG = 0.70   # 70% der Jahreskilometer auf mautpflichtigem Netz

# Jährliche Mautsteigerung (Basis-Szenario aus maut_szenarien_2030.ipynb)
MAUT_STEIGERUNG_DIESEL = 0.05   # ~5% p.a. für Diesel (CO2-Klassendrift + ETS2)
MAUT_STEIGERUNG_BEV    = 0.02   # ~2% p.a. für BEV (nur Infrastrukturteil)

print('Parameter gesetzt:')
print(f'  Kaufpreis Diesel:     {KAUFPREIS["Diesel"]:>10,.0f} €')
print(f'  Kaufpreis BEV:        {KAUFPREIS["BEV"]:>10,.0f} €  (+Infra {KAUFPREIS["Depot_Infrastruktur"]:,.0f} €)')
print(f'  Mehrkosten Anschaffung: {KAUFPREIS["BEV"]+KAUFPREIS["Depot_Infrastruktur"]-KAUFPREIS["Diesel"]:>+,.0f} €')
print(f'  Effektiver Strompreis: {STROM_EFFEKTIV*100:.1f} ct/kWh')
print(f'  Jahreskilometer:      {KM_JAHR:>10,} km')

## 4. TCO-Berechnung

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BERECHNUNGSFUNKTIONEN
# ─────────────────────────────────────────────────────────────────────────────

def jaehrliche_kosten(antrieb, jahr_idx):
    """
    Berechnet alle jährlichen Betriebskosten für ein gegebenes Betriebsjahr.
    jahr_idx: 0 = erstes Betriebsjahr
    """
    # Energiekosten
    if antrieb == 'Diesel':
        preis = ENERGIEPREIS_START['Diesel_EUR_L'] * (1 + PREISSTEIGERUNG['Diesel']) ** jahr_idx
        energie = VERBRAUCH['Diesel_L_100km'] / 100 * KM_JAHR * preis
    else:  # BEV
        preis = STROM_EFFEKTIV * (1 + PREISSTEIGERUNG['Strom']) ** jahr_idx
        energie = VERBRAUCH['BEV_kWh_100km'] / 100 * KM_JAHR * preis

    # Wartungskosten
    wartung = WARTUNG_EUR_KM[antrieb] * KM_JAHR

    # Versicherung
    versicherung = VERSICHERUNG[antrieb]

    # Mautkosten
    maut_ct = MAUT_START_CT_KM[antrieb]
    if antrieb == 'Diesel':
        maut_ct *= (1 + MAUT_STEIGERUNG_DIESEL) ** jahr_idx
    else:
        maut_ct *= (1 + MAUT_STEIGERUNG_BEV) ** jahr_idx
    maut = maut_ct / 100 * KM_JAHR * ANTEIL_MAUTPFLICHTIG

    return {
        'Energie':      energie,
        'Wartung':      wartung,
        'Versicherung': versicherung,
        'Maut':         maut,
        'Gesamt':       energie + wartung + versicherung + maut,
    }


def berechne_tco(antrieb):
    """
    Berechnet den vollständigen TCO-Verlauf über NUTZUNGSJAHRE.
    Gibt DataFrame mit kumulierten und diskontierten Kosten zurück.
    """
    zeilen = []
    kumuliert_undiskontiert = 0
    kumuliert_diskontiert   = 0

    # Anschaffung im Jahr 0 (undiskontiert)
    anschaffung = KAUFPREIS[antrieb]
    if antrieb == 'BEV':
        anschaffung += KAUFPREIS['Depot_Infrastruktur'] - KAUFPREIS['BEV_Foerderung']
    kumuliert_undiskontiert += anschaffung
    kumuliert_diskontiert   += anschaffung

    for j in range(NUTZUNGSJAHRE):
        k = jaehrliche_kosten(antrieb, j)
        diskontfaktor = 1 / (1 + DISKONTIERUNG) ** (j + 1)
        kumuliert_undiskontiert += k['Gesamt']
        kumuliert_diskontiert   += k['Gesamt'] * diskontfaktor

        # Restwert im letzten Jahr abziehen
        restwert = 0
        if j == NUTZUNGSJAHRE - 1:
            restwert = KAUFPREIS[antrieb] * RESTWERT_RATE[antrieb]
            kumuliert_undiskontiert -= restwert
            kumuliert_diskontiert   -= restwert * diskontfaktor

        zeilen.append({
            'Jahr':                  STARTJAHR + j,
            'Betriebsjahr':          j + 1,
            'Energie (€)':           round(k['Energie']),
            'Wartung (€)':           round(k['Wartung']),
            'Versicherung (€)':      round(k['Versicherung']),
            'Maut (€)':              round(k['Maut']),
            'Jährlich gesamt (€)':   round(k['Gesamt']),
            'Kumuliert unkisk. (€)': round(kumuliert_undiskontiert),
            'Kumuliert disk. (€)':   round(kumuliert_diskontiert),
            'Je km (ct)':            round(kumuliert_undiskontiert / ((j+1) * KM_JAHR) * 100, 1),
        })

    return pd.DataFrame(zeilen)


df_diesel = berechne_tco('Diesel')
df_bev    = berechne_tco('BEV')

print('Diesel-TCO (erste und letzte 2 Jahre):')
print(df_diesel[['Betriebsjahr','Energie (€)','Maut (€)','Kumuliert unkisk. (€)','Je km (ct)']]
      .iloc[[0,1,-2,-1]].to_string(index=False))
print()
print('BEV-TCO (erste und letzte 2 Jahre):')
print(df_bev[['Betriebsjahr','Energie (€)','Maut (€)','Kumuliert unkisk. (€)','Je km (ct)']]
      .iloc[[0,1,-2,-1]].to_string(index=False))

## 5. Break-Even-Analyse und Visualisierung

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ABBILDUNG 1: Kumulierte TCO-Kurven + Break-Even
# ─────────────────────────────────────────────────────────────────────────────

# Break-Even finden (undiskontiert)
breakeven_jahr = None
for j in range(NUTZUNGSJAHRE):
    if df_bev.iloc[j]['Kumuliert unkisk. (€)'] <= df_diesel.iloc[j]['Kumuliert unkisk. (€)']:
        breakeven_jahr = j + 1
        break

jahre = df_diesel['Betriebsjahr'].values
# Jahr 0 (Anschaffung) hinzufügen
tco_d = np.concatenate([[KAUFPREIS['Diesel']], df_diesel['Kumuliert unkisk. (€)'].values])
tco_b = np.concatenate([[KAUFPREIS['BEV'] + KAUFPREIS['Depot_Infrastruktur']],
                         df_bev['Kumuliert unkisk. (€)'].values])
x_vals = np.arange(0, NUTZUNGSJAHRE + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Links: Kumulierte TCO ---
ax1 = axes[0]
ax1.plot(x_vals, tco_d / 1e6, color='#d62728', linewidth=2.5, marker='o',
         markersize=5, label='Diesel')
ax1.plot(x_vals, tco_b / 1e6, color='#1f77b4', linewidth=2.5, marker='s',
         markersize=5, linestyle='--', label='E-LKW (BEV)')

if breakeven_jahr:
    be_x = breakeven_jahr
    be_y = df_bev.iloc[breakeven_jahr-1]['Kumuliert unkisk. (€)'] / 1e6
    ax1.axvline(x=be_x, color='green', linestyle=':', linewidth=1.5, alpha=0.7)
    ax1.scatter([be_x], [be_y], color='green', zorder=5, s=120,
               label=f'Break-Even: Jahr {be_x}')
    ax1.annotate(f'Break-Even\nJahr {be_x}', (be_x, be_y),
                textcoords='offset points', xytext=(10, -30),
                fontsize=10, color='green')
else:
    ax1.text(0.5, 0.5, 'Kein Break-Even\ninnerhalb 8 Jahre',
            transform=ax1.transAxes, ha='center', fontsize=11, color='gray')

ax1.set_xlabel('Betriebsjahr')
ax1.set_ylabel('Kumulierte TCO (Mio. €)')
ax1.set_title('Kumulierte Gesamtkosten: Diesel vs. E-LKW', fontweight='bold')
ax1.legend(fontsize=10)
ax1.set_xticks(x_vals)

# --- Rechts: Kostenstruktur im Querschnitt (Jahr 1 und Jahr 8) ---
ax2 = axes[1]
kategorien = ['Energie', 'Wartung', 'Versicherung', 'Maut']
farben = ['#2ca02c', '#ff7f0e', '#9467bd', '#1f77b4']

def jahreskosten_liste(df, j=0):
    row = df.iloc[j]
    return [row['Energie (€)'], row['Wartung (€)'], row['Versicherung (€)'], row['Maut (€)']]

x = np.arange(4)
w = 0.35

d_j1 = jahreskosten_liste(df_diesel, 0)
b_j1 = jahreskosten_liste(df_bev, 0)
d_j8 = jahreskosten_liste(df_diesel, -1)
b_j8 = jahreskosten_liste(df_bev, -1)

bottom_d, bottom_b = 0, 0
bars_d, bars_b = [], []
for i, (k, farbe) in enumerate(zip(kategorien, farben)):
    bd = ax2.bar(0 - w/2, d_j1[i]/1e3, w, bottom=bottom_d/1e3,
                color=farbe, alpha=0.9, label=k if i == 0 else '_')
    ax2.bar(0 + w/2, b_j1[i]/1e3, w, bottom=bottom_b/1e3,
            color=farbe, alpha=0.6, hatch='//')
    ax2.bar(2 - w/2, d_j8[i]/1e3, w, bottom=sum(d_j8[:i])/1e3,
            color=farbe, alpha=0.9)
    ax2.bar(2 + w/2, b_j8[i]/1e3, w, bottom=sum(b_j8[:i])/1e3,
            color=farbe, alpha=0.6, hatch='//')
    bottom_d += d_j1[i]
    bottom_b += b_j1[i]

# Legende
legend_patches = [mpatches.Patch(facecolor=f, label=k) for f, k in zip(farben, kategorien)]
legend_patches += [
    mpatches.Patch(facecolor='gray', alpha=0.9, label='Diesel'),
    mpatches.Patch(facecolor='gray', alpha=0.6, hatch='//', label='E-LKW')
]
ax2.legend(handles=legend_patches, fontsize=8, loc='upper left')
ax2.set_xticks([0-w/2, 0+w/2, 2-w/2, 2+w/2])
ax2.set_xticklabels(['Diesel\nJ.1', 'E-LKW\nJ.1', 'Diesel\nJ.8', 'E-LKW\nJ.8'])
ax2.set_ylabel('Jährliche Betriebskosten (Tsd. €)')
ax2.set_title('Kostenstruktur: Jahr 1 vs. Jahr 8', fontweight='bold')

plt.tight_layout()
plt.savefig('tco_elkw_breakeven.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nBreak-Even: {"Jahr " + str(breakeven_jahr) if breakeven_jahr else "nicht innerhalb 8 Jahre"}')
print(f'TCO Diesel Jahr 8:  {df_diesel.iloc[-1]["Kumuliert unkisk. (€)"]/1e3:>8,.0f} Tsd. €')
print(f'TCO E-LKW  Jahr 8:  {df_bev.iloc[-1]["Kumuliert unkisk. (€)"]/1e3:>8,.0f} Tsd. €')
print(f'Differenz           {(df_diesel.iloc[-1]["Kumuliert unkisk. (€)"]-df_bev.iloc[-1]["Kumuliert unkisk. (€)"])/1e3:>+8,.0f} Tsd. € (+ = BEV günstiger)')

## 6. Sensitivitätsanalyse: Ladestrategie und Kilometerleistung

Zwei kritische Hebel: Wie viel kostet der Strom? Wie viele km fährt der LKW?

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SENSITIVITÄTSANALYSE
# ─────────────────────────────────────────────────────────────────────────────

def tco_total_bev(km_jahr, strom_ct_kwh, nutzungsjahre=8):
    """Berechnet Gesamt-TCO BEV für gegebene Parameter."""
    anschaffung = KAUFPREIS['BEV'] + KAUFPREIS['Depot_Infrastruktur'] - KAUFPREIS['BEV_Foerderung']
    total = anschaffung
    for j in range(nutzungsjahre):
        strom_preis = (strom_ct_kwh / 100) * (1 + PREISSTEIGERUNG['Strom']) ** j
        energie = VERBRAUCH['BEV_kWh_100km'] / 100 * km_jahr * strom_preis
        wartung = WARTUNG_EUR_KM['BEV'] * km_jahr
        maut = MAUT_START_CT_KM['BEV'] * (1 + MAUT_STEIGERUNG_BEV)**j / 100 * km_jahr * ANTEIL_MAUTPFLICHTIG
        total += energie + wartung + VERSICHERUNG['BEV'] + maut
    total -= KAUFPREIS['BEV'] * RESTWERT_RATE['BEV']
    return total


def tco_total_diesel(km_jahr, nutzungsjahre=8):
    """Berechnet Gesamt-TCO Diesel für gegebene Parameter."""
    total = KAUFPREIS['Diesel']
    for j in range(nutzungsjahre):
        preis = ENERGIEPREIS_START['Diesel_EUR_L'] * (1 + PREISSTEIGERUNG['Diesel']) ** j
        energie = VERBRAUCH['Diesel_L_100km'] / 100 * km_jahr * preis
        wartung = WARTUNG_EUR_KM['Diesel'] * km_jahr
        maut = MAUT_START_CT_KM['Diesel'] * (1 + MAUT_STEIGERUNG_DIESEL)**j / 100 * km_jahr * ANTEIL_MAUTPFLICHTIG
        total += energie + wartung + VERSICHERUNG['Diesel'] + maut
    total -= KAUFPREIS['Diesel'] * RESTWERT_RATE['Diesel']
    return total


# Grid: Strompreis vs. Jahreskilometer
strom_werte = np.arange(10, 50, 2)   # 10–48 ct/kWh
km_werte    = np.arange(60_000, 161_000, 10_000)  # 60k–160k km/Jahr

# Differenzmatrix: TCO_Diesel - TCO_BEV (positiv = BEV günstiger)
diff_matrix = np.zeros((len(km_werte), len(strom_werte)))
for i, km in enumerate(km_werte):
    for j, strom in enumerate(strom_werte):
        diff_matrix[i, j] = (tco_total_diesel(km) - tco_total_bev(km, strom)) / 1000  # in Tsd. €

fig, ax = plt.subplots(figsize=(12, 6))
cmap = plt.cm.RdYlGn  # Rot (Diesel günstiger) → Grün (BEV günstiger)
im = ax.imshow(diff_matrix, aspect='auto', cmap=cmap,
               vmin=-100, vmax=100,
               extent=[strom_werte[0]-1, strom_werte[-1]+1,
                       km_werte[-1]+5000, km_werte[0]-5000])

# Nulllinie (Break-Even-Linie)
ax.contour(strom_werte, km_werte, diff_matrix, levels=[0],
           colors=['black'], linewidths=2.5)

plt.colorbar(im, ax=ax, label='TCO-Vorteil BEV (Tsd. €, 8 Jahre, + = BEV günstiger)')

# Referenzpunkte
for label, strom, km in [
    ('Depot PV\n(8 ct)',       8,  120_000),
    ('Industrie\n(16 ct)',    16,  100_000),
    ('Gewerbe\n(22 ct)',      22,   80_000),
    ('Öffentlich\n(45 ct)',   45,  100_000),
]:
    ax.scatter([strom], [km], color='navy', s=80, zorder=5)
    ax.annotate(label, (strom, km), textcoords='offset points',
               xytext=(5, 5), fontsize=8, color='navy')

ax.set_xlabel('Effektiver Strompreis (ct/kWh)')
ax.set_ylabel('Jahreskilometerleistung (km)')
ax.set_title('Break-Even-Heatmap: Wann ist E-LKW günstiger als Diesel? (8 Jahre TCO)',
             fontweight='bold')
ax.text(12, 155_000, 'E-LKW günstiger →', color='darkgreen', fontsize=10, fontstyle='italic')
ax.text(35, 70_000, '← Diesel günstiger', color='darkred', fontsize=10, fontstyle='italic')

plt.tight_layout()
plt.savefig('tco_heatmap_breakeven.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Szenarien: Best Case / Base Case / Worst Case

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SZENARIO-VERGLEICH
# ─────────────────────────────────────────────────────────────────────────────
szenarien = {
    'Diesel (Referenz)': {
        'km':    100_000,
        'strom': None,
        'antrieb': 'Diesel',
        'beschreibung': '100.000 km/Jahr, Diesel 1,50 €/L, steigende Maut'
    },
    'BEV Best Case\n(Depot + PV)': {
        'km':    120_000,
        'strom': 10,
        'antrieb': 'BEV',
        'beschreibung': '120.000 km/Jahr, 10 ct/kWh Eigenverbrauch PV'
    },
    'BEV Base Case\n(Depot Industrie)': {
        'km':    100_000,
        'strom': 18,
        'antrieb': 'BEV',
        'beschreibung': '100.000 km/Jahr, 18 ct/kWh Industriestrom'
    },
    'BEV Worst Case\n(öffentl. Laden)': {
        'km':     80_000,
        'strom': 45,
        'antrieb': 'BEV',
        'beschreibung': '80.000 km/Jahr, 45 ct/kWh öffentliche Ladung'
    },
}

ergebnisse = {}
for name, s in szenarien.items():
    if s['antrieb'] == 'Diesel':
        tco = tco_total_diesel(s['km'])
    else:
        tco = tco_total_bev(s['km'], s['strom'])
    tco_km = tco / (s['km'] * NUTZUNGSJAHRE) * 100  # ct/km
    ergebnisse[name] = {'TCO gesamt (€)': round(tco), 'TCO je km (ct)': round(tco_km, 1)}

print('Szenariovergleich (8 Jahre, alle Kosten inkl. Anschaffung):')
print(f'{"":<35} {"TCO gesamt":>15} {"je km (ct)":>12}')
print('─' * 65)
diesel_tco = ergebnisse['Diesel (Referenz)']['TCO gesamt (€)']
for name, r in ergebnisse.items():
    diff = r['TCO gesamt (€)'] - diesel_tco
    diff_str = f'({diff:+,.0f} €)' if name != 'Diesel (Referenz)' else ''
    clean = name.replace('\n', ' ')
    print(f'{clean:<35} {r["TCO gesamt (€)"]:>12,.0f} €  {r["TCO je km (ct)"]:>6.1f} ct  {diff_str}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ABBILDUNG 3: Szenario-Balkendiagramm
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

namen  = [n.replace('\n', ' ') for n in ergebnisse.keys()]
werte  = [v['TCO gesamt (€)'] / 1e3 for v in ergebnisse.values()]
farben = ['#d62728', '#2ca02c', '#1f77b4', '#ff7f0e']

bars = ax.barh(namen, werte, color=farben, edgecolor='white')
for bar, val in zip(bars, werte):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f} Tsd. €', va='center', fontsize=10)

ax.axvline(x=werte[0], color='red', linestyle='--', linewidth=1.5,
           alpha=0.5, label='Diesel-Referenz')
ax.set_xlabel('TCO über 8 Jahre (Tsd. €)')
ax.set_title('Szenariovergleich: E-LKW vs. Diesel (TCO 8 Jahre)', fontweight='bold')
ax.set_xlim(0, max(werte) * 1.15)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('tco_szenarien_vergleich.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EXPORT
# ─────────────────────────────────────────────────────────────────────────────
df_export = pd.DataFrame([
    {'Antrieb': 'Diesel', **row}
    for _, row in df_diesel.iterrows()
] + [
    {'Antrieb': 'BEV', **row}
    for _, row in df_bev.iterrows()
])
df_export.to_csv('tco_elkw_ergebnisse.csv', index=False, sep=';', decimal=',')
print('Ergebnisse exportiert: tco_elkw_ergebnisse.csv')
print()
print('Wichtige Quellen:')
print('  ICCT TCO-Studie 2023: https://theicct.org')
print('  TNO-Bericht 2022:     https://www.agora-verkehrswende.de')
print('  MAN eTruck Daten:     https://www.man.eu/de/lkw/elektro-lkw')
print('  BDEW Strompreise:     https://www.bdew.de')
print('  T&E Depot Charging:   https://www.transportenvironment.org')